In [1]:
import os

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, recall_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import pickle

### Loading and pre-processing the data

In [2]:
FILENAME = os.path.join('..', 'data', 'diabetes_012_health_indicators_BRFSS2015.csv')

data = pd.read_csv(FILENAME)
data.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [4]:
# standardizing column headers to lower case
data.columns = data.columns.str.lower()

In [5]:
target_column = 'diabetes_012'

data[target_column].value_counts(normalize=True)

diabetes_012
0.0    0.842412
2.0    0.139333
1.0    0.018255
Name: proportion, dtype: float64

In [6]:
#The proportion of prediabeis patients is less than 2%. Merging prediabetes with non-diabetes to focus the model on identifying diabetes patients.
data[target_column] = data[target_column].replace({1:0})
data[target_column] = data[target_column].replace({2:1})

data[target_column].value_counts(normalize=True)


diabetes_012
0.0    0.860667
1.0    0.139333
Name: proportion, dtype: float64

### Training

#### Decision Tree Classifier

In [7]:
X = data.drop(columns=[target_column], axis=1)
y = data[target_column]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



In [8]:
### Training decision tree classifier base model
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

y_pred = dt_classifier.predict(X_test)

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred))
print("Recall Score:", recall_score(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

         0.0       0.89      0.87      0.88     43667
         1.0       0.30      0.33      0.31      7069

    accuracy                           0.80     50736
   macro avg       0.59      0.60      0.60     50736
weighted avg       0.81      0.80      0.80     50736

Confusion Matrix:
 [[38129  5538]
 [ 4719  2350]]
ROC AUC Score: 0.6028069716907356
Recall Score: 0.33243740274437683


In [9]:
# Training decision tree classifier to handle class imbalance and introducing grid search for hyperparameter tuning
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))
print("Class Weights:", class_weight_dict)

dt_classifier_balanced = DecisionTreeClassifier(random_state=42, class_weight=class_weight_dict)

param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

kfold = KFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(estimator=dt_classifier_balanced, 
                           param_grid=param_grid, 
                           cv=kfold, 
                           scoring='recall', 
                           n_jobs=-1,
                           verbose=1)

grid_search.fit(X_train, y_train)
best_dt_classifier = grid_search.best_estimator_
y_pred_balanced = best_dt_classifier.predict(X_test)

print("Classification Report:\n", classification_report(y_test, y_pred_balanced))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_balanced))
print("Recall Score:", recall_score(y_test, y_pred_balanced))

Class Weights: {np.float64(0.0): np.float64(0.5809454562109614), np.float64(1.0): np.float64(3.5884994872157585)}
Fitting 3 folds for each of 45 candidates, totalling 135 fits
Classification Report:
               precision    recall  f1-score   support

         0.0       0.95      0.69      0.80     43667
         1.0       0.29      0.79      0.43      7069

    accuracy                           0.70     50736
   macro avg       0.62      0.74      0.61     50736
weighted avg       0.86      0.70      0.75     50736

Confusion Matrix:
 [[30137 13530]
 [ 1495  5574]]
ROC AUC Score: 0.739334131874599
Recall Score: 0.7885132267647474


#### Random forest classifier after handling class imbalance

In [10]:
rf = RandomForestClassifier(random_state=42,
                             class_weight=class_weight_dict,
                             n_jobs=-1)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15, 20, None], 
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_rf = GridSearchCV(estimator=rf, 
                              param_grid=param_grid_rf,
                              cv=kfold, 
                              scoring='recall', 
                              n_jobs=-1,
                              verbose=2)

grid_search_rf.fit(X_train, y_train)
best_rf_classifier = grid_search_rf.best_estimator_
y_pred_rf = best_rf_classifier.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_rf))
print("Recall Score:", recall_score(y_test, y_pred_rf))

Fitting 3 folds for each of 90 candidates, totalling 270 fits
Classification Report:
               precision    recall  f1-score   support

         0.0       0.95      0.70      0.81     43667
         1.0       0.30      0.77      0.43      7069

    accuracy                           0.71     50736
   macro avg       0.62      0.74      0.62     50736
weighted avg       0.86      0.71      0.76     50736

Confusion Matrix:
 [[30762 12905]
 [ 1633  5436]]
ROC AUC Score: 0.7367296377994776
Recall Score: 0.7689913707738011


#### XGBoost classifier

In [11]:
pos= sum(y_train==1)
neg= sum(y_train==0)
scale_pos_weight = neg / pos
xgb = XGBClassifier(random_state=42,
                      scale_pos_weight=scale_pos_weight,
                      use_label_encoder=False,
                      eval_metric='logloss',
                      n_jobs=-1)

param_grid_xgb = {
    'n_estimators': [100, 200, 400],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

grid_search_xgb = GridSearchCV(estimator=xgb,
                               param_grid=param_grid_xgb,
                               scoring='recall',
                               cv=kfold,
                               n_jobs=-1,
                               verbose=2)
grid_search_xgb.fit(X_train, y_train)
best_xgb_classifier = grid_search_xgb.best_estimator_
y_pred_xgb = best_xgb_classifier.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_xgb))
print("Recall Score:", recall_score(y_test, y_pred_xgb))

Fitting 3 folds for each of 972 candidates, totalling 2916 fits


c:\jack\study\e2eprojects\mlz2025_mtp\mlz\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
c:\jack\study\e2eprojects\mlz2025_mtp\mlz\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:45:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.71      0.82     43667
         1.0       0.31      0.79      0.44      7069

    accuracy                           0.72     50736
   macro avg       0.63      0.75      0.63     50736
weighted avg       0.86      0.72      0.76     50736

Confusion Matrix:
 [[31048 12619]
 [ 1457  5612]]
ROC AUC Score: 0.7524531417237732
Recall Score: 0.7938888102984863


In [15]:
xgb_tuned = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    n_estimators=best_xgb_classifier.n_estimators,
    learning_rate=best_xgb_classifier.learning_rate,
    max_depth=best_xgb_classifier.max_depth,
    subsample=best_xgb_classifier.subsample,
    colsample_bytree=best_xgb_classifier.colsample_bytree,
    min_child_weight=best_xgb_classifier.min_child_weight
)

scale_pos_wt_grid = {
    'scale_pos_weight': [scale_pos_weight, scale_pos_weight**1.5, scale_pos_weight**2],
}

grid_search_xgb_tuned = GridSearchCV(estimator=xgb_tuned,
                               param_grid=scale_pos_wt_grid,
                               scoring='recall',
                               cv=kfold,
                               n_jobs=-1,
                               verbose=2)
grid_search_xgb_tuned.fit(X_train, y_train)
best_xgb_classifier_scale_pos = grid_search_xgb_tuned.best_estimator_
y_pred_xgb_tuned = best_xgb_classifier_scale_pos.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_xgb_tuned))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb_tuned))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_xgb_tuned))
print("Recall Score:", recall_score(y_test, y_pred_xgb_tuned))

Fitting 3 folds for each of 3 candidates, totalling 9 fits


c:\jack\study\e2eprojects\mlz2025_mtp\mlz\Lib\site-packages\xgboost\training.py:199: UserWarning: [14:08:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Classification Report:
               precision    recall  f1-score   support

         0.0       0.99      0.35      0.52     43667
         1.0       0.20      0.97      0.33      7069

    accuracy                           0.44     50736
   macro avg       0.59      0.66      0.42     50736
weighted avg       0.88      0.44      0.49     50736

Confusion Matrix:
 [[15332 28335]
 [  188  6881]]
ROC AUC Score: 0.6622584156771578
Recall Score: 0.9734050077804498


Looks like the XGBoost model with tuned scale_pos_weight gives the best recall score among all models trained. 
However, this comes at a cost of lower precision. Depending on the application, this trade-off may be acceptable 
or not. I'm going to choose the model with scale_pos_weight=scale_pos_weight as the final model for deployment.

In [19]:
final_model = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    n_estimators=best_xgb_classifier.n_estimators,
    learning_rate=best_xgb_classifier.learning_rate,
    max_depth=best_xgb_classifier.max_depth,
    subsample=best_xgb_classifier.subsample,
    colsample_bytree=best_xgb_classifier.colsample_bytree,
    min_child_weight=best_xgb_classifier.min_child_weight,
    scale_pos_weight=scale_pos_weight
)

final_model.fit(X_train, y_train)
    
y_pred_final = final_model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_final))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_final))
print("Recall Score:", recall_score(y_test, y_pred_final))

c:\jack\study\e2eprojects\mlz2025_mtp\mlz\Lib\site-packages\xgboost\training.py:199: UserWarning: [14:17:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Classification Report:
               precision    recall  f1-score   support

         0.0       0.96      0.71      0.82     43667
         1.0       0.31      0.79      0.44      7069

    accuracy                           0.72     50736
   macro avg       0.63      0.75      0.63     50736
weighted avg       0.86      0.72      0.76     50736

Confusion Matrix:
 [[31048 12619]
 [ 1457  5612]]
ROC AUC Score: 0.7524531417237732
Recall Score: 0.7938888102984863


In [20]:
print(f'n_estimators, {best_xgb_classifier.n_estimators},')
print(f'learning_rate, {best_xgb_classifier.learning_rate},')
print(f'max_depth, {best_xgb_classifier.max_depth},')
print(f'subsample, {best_xgb_classifier.subsample},')
print(f'colsample_bytree, {best_xgb_classifier.colsample_bytree},')
print(f'min_child_weight, {best_xgb_classifier.min_child_weight},')
print(f'scale_pos_weight, {best_xgb_classifier.scale_pos_weight},')


n_estimators, 100,
learning_rate, 0.2,
max_depth, 3,
subsample, 0.7,
colsample_bytree, 1.0,
min_child_weight, 1,
scale_pos_weight, 6.176998974431517,


#### Saving the final model

In [23]:
with open('model.bin', 'wb') as f_out:
    pickle.dump(final_model, f_out)

In [ ]:
def predict_single(patient_dict: dict) -> float:
    # convert dict -> single-row DataFrame
    df = pd.DataFrame([patient_dict])

    # try to align columns with training order if available
    feature_names = getattr(final_model, "feature_names_in_", None)
    if feature_names is not None:
        df = df.loc[:, feature_names]

    prob = final_model.predict_proba(df)[0, 1]
    return float(prob)

In [ ]:
patient_example = {
  "highbp": 1,
  "highchol": 1,
  "cholcheck": 1,
  "bmi": 40,
  "smoker": 1,
  "stroke": 0,
  "heartdiseaseorattack": 0,
  "physactivity": 0,
  "fruits": 0,
  "veggies": 1,
  "hvyalcoholconsump": 0,
  "anyhealthcare": 1,
  "nodocbccost": 0,
  "genhlth": 5,
  "menthlth": 18,
  "physhlth": 15,
  "diffwalk": 1,
  "sex": 0,
  "age": 9,
  "education": 4,
  "income": 3
}

In [ ]:
predict_single(patient_example)

In [ ]:
# 1,1,1,30,1,0,1,0,1,1,0,1,0,5,30,30,1,0,9,5,1

In [ ]:
patient_example = {
  "highbp": 1,
  "highchol": 1,
  "cholcheck": 1,
  "bmi": 30,
  "smoker": 1,
  "stroke": 0,
  "heartdiseaseorattack": 1,
  "physactivity": 0,
  "fruits": 1,
  "veggies": 1,
  "hvyalcoholconsump": 0,
  "anyhealthcare": 1,
  "nodocbccost": 0,
  "genhlth": 5,
  "menthlth": 30,
  "physhlth": 30,
  "diffwalk": 1,
  "sex": 0,
  "age": 9,
  "education": 1,
  "income": 1
}

In [ ]:
# 1,1,1,21,0,0,0,1,1,1,0,1,0,3,0,0,0,0,10,4,3

In [ ]:
patient_example = {
  "highbp": 1,
  "highchol": 1,
  "cholcheck": 1,
  "bmi": 21,
  "smoker": 0,
  "stroke": 0,
  "heartdiseaseorattack": 0,
  "physactivity": 1,
  "fruits": 1,
  "veggies": 1,
  "hvyalcoholconsump": 0,
  "anyhealthcare": 1,
  "nodocbccost": 0,
  "genhlth": 3,
  "menthlth": 0,
  "physhlth": 0,
  "diffwalk": 0,
  "sex": 0,
  "age": 10,
  "education": 4,
  "income": 3
}